In [97]:
import os
from langchain_cerebras import ChatCerebras
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from pydantic import BaseModel
from pymongo import MongoClient
import sib_api_v3_sdk
from sib_api_v3_sdk.rest import ApiException
from langchain.agents import create_agent
from langchain_groq import ChatGroq

load_dotenv()

CEREBRAS_API_KEY=os.getenv("CEREBRAS_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
BREVO_API_KEY=os.getenv("BREVO_API_KEY")
MONGODB_URI=os.getenv("MONGODB_URI")


In [98]:
DB_NAME = "soulsync"
COLLECTION_NAME="users"
SENDER_NAME="SoulSync"
SENDER_MAIL="lokeshvijayraina@gmail.com"


In [99]:
llm_gpt=ChatGroq(model="openai/gpt-oss-120b",api_key=GROQ_API_KEY)
client= MongoClient(MONGODB_URI)
users_collection=client[DB_NAME][COLLECTION_NAME]

In [100]:
class EmailTemplate(BaseModel):
    subject_template: str
    body_template: str
    reason: str

class EmailDraft(BaseModel):
    recipient: str
    subject: str
    body: str

class DispatchedResult(BaseModel):
    totalUsers: int
    sent: int
    failed: int


In [101]:
@tool
def find_users(
    filters: dict = None,
    sort_by: str = None,
    ascending: bool = False,
    limit: int = 5,
):
    """
    Query SoulSync users with optional filters, sorting, and limit.

    Available fields:
    - name
    - email
    - totalListeningTime
    - updatedAt (last active)
    - createdAt (joined date)
    """

    query = filters or {}

    targeted_users = users_collection.find(
        query,
        {
            "_id": 0,
            "name": 1,
            "email": 1,
            "totalListeningTime": 1,
            "updatedAt": 1,
            "createdAt": 1,
        },
    )

    if sort_by:
        targeted_users = targeted_users.sort(
            sort_by,
            -1 if not ascending else 1,
        )

    targeted_users = targeted_users.limit(limit)
    
    def serialize(user):
        return {
            k: (v.isoformat() if hasattr(v, "isoformat") else v)
            for k, v in user.items()
        }

    return [serialize(u) for u in targeted_users]


In [102]:
finder_agent=create_agent(
    model=llm_gpt,
    tools=[find_users],
    system_prompt=
    """
    You are LookOut's user discovery agent.

Your job is to identify the correct SoulSync users by calling the `find_users` tool.

Available fields:

* `name`
* `email`
* `country`
* `totalListeningTime` (seconds)
* `createdAt` (join date)
* `updatedAt` (last active where the time is in the UTC soo add 5:30 and show in the indian railway time)

Infer these arguments from the user's request:

* `filters`
* `sort_by`
* `ascending`
* `limit`

Intent examples:

* Top / most active → `sort_by="totalListeningTime"`, `ascending=False`
* Least active → `sort_by="totalListeningTime"`, `ascending=True`
* Newest → `sort_by="createdAt"`, `ascending=False`
* Oldest → `sort_by="createdAt"`, `ascending=True`
* Recently active → `sort_by="updatedAt"`, `ascending=False`
* Inactive → `sort_by="updatedAt"`, `ascending=True`
* Country-specific requests → use `filters`

Rules:

* Always call `find_users`.
* Never invent users or fields.
* Combine filters and sorting when needed.
* If the request cannot be answered using the available fields, explain why.

    """)

In [103]:
template_llm = llm_gpt.with_structured_output(EmailTemplate)

def generate_template(sample_user: dict, intent: str) -> EmailTemplate:
    available = list(sample_user.keys())
    prompt = f"""
Product: SoulSync
User intent: {intent}
Available placeholder fields (use ONLY these exact names, nothing else): {available}
Write ONE email template. Use placeholders like {{{available[0]}}}, {{{available[1]}}}, etc.
Do NOT invent any fields not in the list above.
Sign off as "The SoulSync Team". Keep body under 100 words.
"""
    return template_llm.invoke(prompt)


In [104]:
class SafeDict(dict):
    """Returns '{key}' for any missing key instead of raising KeyError."""
    def __missing__(self, key):
        return f"{{{key}}}"

def renderTemplate(template: EmailTemplate, user: dict) -> EmailDraft:
    subject = template.subject_template.format_map(SafeDict(**user))
    body = template.body_template.format_map(SafeDict(**user))
    return EmailDraft(recipient=user["email"], subject=subject, body=body)


In [105]:
@tool
def sendMail(receiver: str, subject: str, body: str) -> str:
    """Send an approved email using Brevo."""
    configuration = sib_api_v3_sdk.Configuration()
    configuration.api_key["api-key"] = BREVO_API_KEY
    api_instance = sib_api_v3_sdk.TransactionalEmailsApi(sib_api_v3_sdk.ApiClient(configuration))
    send_smtp_email = sib_api_v3_sdk.SendSmtpEmail(
        sender={"name": SENDER_NAME, "email": SENDER_MAIL},
        to=[{"email": receiver}],
        subject=subject,
        html_content=body,
    )
    try:
        api_response = api_instance.send_transac_email(send_smtp_email)
        return f"Sent to {receiver} (id: {api_response.message_id})"
    except ApiException as e:
        return f"Failed: {e}"

In [106]:
import json

def get_tool_result(agent_response):
    for msg in agent_response["messages"]:
        if hasattr(msg, "name") and msg.name == "find_users":
            return json.loads(msg.content)
    return None


In [107]:
def run_dispatch(user_prompt: str):
    response = finder_agent.invoke({"messages": [HumanMessage(content=user_prompt)]})
    matched_users = get_tool_result(response)

    for i, user in enumerate(matched_users):
        user["minutesListened"] = round(user.pop("totalListeningTime", 0) / 60)
        user["rank"] = i + 1

    template = generate_template(matched_users[0], user_prompt)
    preview = renderTemplate(template, matched_users[0])

    print(f"Matched {len(matched_users)} users.")
    print(f"Subject: {preview.subject}\n\n{preview.body}\n")
    choice = input("Approve this template for all matched users? (y/n): ").strip().lower()

    if choice != "y":
        print("Aborted.")
        return DispatchedResult(totalUsers=len(matched_users), sent=0, failed=len(matched_users))

    sent, failed = 0, 0
    for user in matched_users:
        draft = renderTemplate(template, user)
        result = sendMail.invoke({"receiver": draft.recipient, "subject": draft.subject, "body": draft.body})
        if "Failed" in result:
            failed += 1
        else:
            sent += 1

    return DispatchedResult(totalUsers=len(matched_users), sent=sent, failed=failed)


In [108]:
result = run_dispatch("mail our the top 1 listenedtime user")
print(result)

Matched 1 users.
Subject: Congratulations Loki! You're #1 on SoulSync 🎧

Hi Loki,

Your dedication shines! With 7878 minutes listened, you’ve secured the top spot (rank 1) on SoulSync. Keep the rhythm alive.

Thank you for being an amazing part of our community.

The SoulSync Team

Aborted.
totalUsers=1 sent=0 failed=1
